# 04 — Random Forest Modeling & Validation Protocol

**Owner:** Himanshu Pandey · **Phase 1, Step 4**

This notebook establishes the primary tree-ensemble benchmark for the Water Quality Prediction project (CSAI Project 18).

It implements, evaluates, and diagnoses **Random Forest Regression** using the shared, area-grouped split (`data/splits/splits.parquet`), train-only preprocessing pipeline (`src/preprocess.py`), and feature engineering suite (`src/features.py`).

### Methodological Standards
1. **Regime 1 vs. Regime 2 Governance**:
   - **Regime 1 (Formula Emulation / Benchmark Surrogate)**: Models receive the 8 Ireland chemical inputs (`Ammonia`, `BOD`, `DO`, `Orthophosphate`, `pH`, `Temperature`, `Nitrogen`, `Nitrate`). Because the CCME Water Quality Index is a deterministic mathematical function of these inputs, Regime 1 strictly benchmarks how accurately tree ensembles can approximate the CCME non-linear algebraic equation. It is **never** characterized as independent environmental forecasting.
   - **Regime 2 (Independent Environmental Prediction)**: Strictly excludes all chemical measurements and derived features. Models predict water quality solely from environmental metadata and cyclic spatiotemporal covariates (`Country`, `Waterbody Type`, `sin_day`, `cos_day`, `month`).
2. **Zero-Leakage Discipline**:
   - Categorical encodings (`Country`, `Waterbody Type`) and numeric medians are fit strictly on training data.
   - All hyperparameter decisions are made **exclusively on the validation partition**.
   - The holdout test partition is evaluated **strictly once** on the single selected model.
3. **Scientific Diagnosis**:
   - Explains Bagging and variance reduction (Breiman, 2001).
   - Compares Mean Decrease in Impurity (MDI / Gini) with Permutation Feature Importance, referencing the known continuous/collinear feature bias identified by Strobl et al. (2007).
   - Evaluates the impact of the **35.92% target ceiling** at $\text{CCME} = 100.0$ via partitioned residual analysis.


In [1]:
import sys
sys.path.insert(0, '..')

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance

from src.config import FIGURES, SEED, TARGET_COLUMN
from src.features import get_regime_splits
from src.models_rf import train, compute_metrics, CategoricalEncoder, apply_subsample

# Global styling & plotting configuration
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
FIGURES.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', category=UserWarning)

print(f"Seed: {SEED} | Target: {TARGET_COLUMN}")
print(f"Figures target directory: {FIGURES}")


Seed: 42 | Target: CCME_Values
Figures target directory: /Users/himanshupandey/Desktop/water_quality_prediction/reports/figures


## 1. Theoretical Foundation: Bagging and Variance Reduction

A Random Forest (Breiman, 2001) is an ensemble of $B$ deep, unpruned decision trees grown on bootstrap resamples $Z^{*b}$ of the training data.

### 1.1 The Bias-Variance Tradeoff
For any estimator $\hat{f}(x)$, the expected generalization mean squared error decomposes into:
$$\mathbb{E}[(y - \hat{f}(x))^2] = \text{Bias}^2(\hat{f}(x)) + \text{Var}(\hat{f}(x)) + \sigma_{\text{irred}}^2$$

An individual deep decision tree has low bias but high variance: small perturbations in the training dataset lead to drastically different split points.

### 1.2 Variance Reduction via Averaging
Let $T(x; \Theta_b)$ denote the prediction of the $b$-th tree. The ensemble predictor is:
$$\hat{f}_{\text{rf}}(x) = \frac{1}{B} \sum_{b=1}^B T(x; \Theta_b)$$

If each individual tree has variance $\sigma^2$ and the pairwise correlation between trees is $\rho = \text{Corr}(T(x; \Theta_i), T(x; \Theta_j))$, the variance of the ensemble average is:
$$\text{Var}(\hat{f}_{\text{rf}}(x)) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

As the number of trees $B \to \infty$, the second term $\frac{1 - \rho}{B}\sigma^2 \to 0$. The asymptotic variance floor is bounded by $\rho \sigma^2$.

### 1.3 Random Feature Subspace (Decorrelation)
To minimize the remaining variance $\rho \sigma^2$, Breiman introduced **random feature selection at each node** ($\text{max\_features} = m \le p$). By forcing each split to choose from a random subset of candidates (typically $m = \sqrt{p}$ or $p/3$), trees are decorrelated, lowering $\rho$ without significantly increasing individual tree bias.

### 1.4 Out-of-Bag (OOB) Generalization Estimation
Because each tree is fit on a bootstrap sample of size $N$ drawn with replacement, the probability that a given observation is omitted from tree $b$ is:
$$\lim_{N \to \infty} \left(1 - \frac{1}{N}\right)^N = \frac{1}{e} \approx 0.3679 \quad (36.8\%)$$
Aggregating predictions for each observation $i$ only across trees where $i$ was out-of-bag yields the **Out-of-Bag (OOB) score**, providing an intrinsic cross-validation estimate without fitting additional models.


## 2. Baseline Model Execution (Regime 1: 8/8 Emulation, 100k Subsample)

We instantiate the baseline model on a reproducible 100,000-sample training partition (`subsample=100_000`, seed=42) using `src/models_rf.py`.

The model uses:
- `n_estimators = 100` trees
- `max_depth = 15`
- `min_samples_leaf = 5`
- `max_features = 'sqrt'`
- `oob_score = True`
- `feature_set = 'engineered'` (23 features: 8 chemical inputs, 10 derived ratios/logs, 2 metadata, 3 temporal)


In [2]:
# Execute baseline Random Forest training via src/models_rf.py
baseline_res = train(
    regime=1,
    feature_set="engineered",
    partial_scenario="8/8",
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    max_features="sqrt",
    subsample=100_000,
    oob_score=True,
    save_reports=True,
    n_jobs=-1,
)

metrics_summary = pd.DataFrame([
    {
        "Partition": "Validation (415,805 rows)",
        "MAE": baseline_res["val_metrics"]["mae"],
        "RMSE": baseline_res["val_metrics"]["rmse"],
        "R²": baseline_res["val_metrics"]["r2"],
    },
    {
        "Partition": "Out-of-Bag (OOB Train)",
        "MAE": np.nan,
        "RMSE": np.nan,
        "R²": baseline_res["oob_score"],
    }
])

print(f"=== Random Forest Baseline Results ({baseline_res['config']['experiment_label']}) ===")
print(f"Features ({baseline_res['config']['n_features']}):")
for i, col in enumerate(baseline_res["feature_names"], 1):
    print(f"  {i:2d}. {col}")
print("\nValidation Performance:")
print(metrics_summary.to_string(index=False))


=== Random Forest Baseline Results (surrogate_emulator_benchmark (formula approximation, NOT independent prediction)) ===
Features (38):
   1. Ammonia (mg/l)
   2. Biochemical Oxygen Demand (mg/l)
   3. Dissolved Oxygen (mg/l)
   4. Orthophosphate (mg/l)
   5. pH (ph units)
   6. Temperature (cel)
   7. Nitrogen (mg/l)
   8. Nitrate (mg/l)
   9. N_P_ratio
  10. Inorganic_N_ratio
  11. Organic_N_estimate
  12. Organic_N_fraction
  13. BOD_DO_ratio
  14. log_Ammonia
  15. log_BOD
  16. log_Orthophosphate
  17. log_Nitrate
  18. log_Nitrogen
  19. sin_day
  20. cos_day
  21. month
  22. Country_Canada
  23. Country_China
  24. Country_England
  25. Country_Ireland
  26. Country_USA
  27. Waterbody Type_Bay
  28. Waterbody Type_Canal
  29. Waterbody Type_Coastal
  30. Waterbody Type_Drainage
  31. Waterbody Type_Effluent
  32. Waterbody Type_Estuarine
  33. Waterbody Type_Lake
  34. Waterbody Type_Marine
  35. Waterbody Type_River
  36. Waterbody Type_Sea Water
  37. Waterbody Type_Sewage


## 3. Hyperparameter Tuning on Validation Split Only

We explore the hyperparameter space across tree depth (`max_depth`), leaf regularization (`min_samples_leaf`), and feature fraction (`max_features`).

### Strict Scientific Guarantee
- **Training partition (`X_train`)**: Fits the trees.
- **Validation partition (`X_val`)**: Evaluates candidate models to pick the best configuration.
- **Test partition (`X_test`)**: Held out completely. It is **never** evaluated during this search.


In [3]:
# Validation tuning grid
param_candidates = [
    {"name": "Shallow (Depth 10, Leaf 5)", "params": {"n_estimators": 50, "max_depth": 10, "min_samples_leaf": 5, "max_features": "sqrt"}},
    {"name": "Medium (Depth 15, Leaf 5)", "params": {"n_estimators": 50, "max_depth": 15, "min_samples_leaf": 5, "max_features": "sqrt"}},
    {"name": "Deep (Depth 20, Leaf 2)", "params": {"n_estimators": 50, "max_depth": 20, "min_samples_leaf": 2, "max_features": "sqrt"}},
    {"name": "Half-Features (Depth 15, m=0.5)", "params": {"n_estimators": 50, "max_depth": 15, "min_samples_leaf": 5, "max_features": 0.5}},
]

tune_records = []
for candidate in param_candidates:
    p = candidate["params"]
    res = train(
        regime=1,
        feature_set="engineered",
        partial_scenario="8/8",
        n_estimators=p["n_estimators"],
        max_depth=p["max_depth"],
        min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"],
        subsample=50_000,
        save_reports=False,
        n_jobs=-1,
    )
    tune_records.append({
        "Configuration": candidate["name"],
        "Max Depth": p["max_depth"],
        "Min Samples Leaf": p["min_samples_leaf"],
        "Max Features": str(p["max_features"]),
        "Val MAE": res["val_metrics"]["mae"],
        "Val RMSE": res["val_metrics"]["rmse"],
        "Val R²": res["val_metrics"]["r2"],
    })

df_tune = pd.DataFrame(tune_records)
print("Validation Tuning Results:")
print(df_tune.to_string(index=False))

# Plot Validation RMSE comparison
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=df_tune, x="Configuration", y="Val RMSE", palette="Blues_r", ax=ax, hue="Configuration", legend=False)
ax.set_title("Validation RMSE across Hyperparameter Configurations", fontweight="bold")
ax.set_ylabel("Validation RMSE (lower is better)")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "rf_hyperparameter_tuning.png", dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved figure: {FIGURES / 'rf_hyperparameter_tuning.png'}")


Validation Tuning Results:
                  Configuration  Max Depth  Min Samples Leaf Max Features  Val MAE  Val RMSE   Val R²
     Shallow (Depth 10, Leaf 5)         10                 5         sqrt 0.935509  1.578838 0.992759
      Medium (Depth 15, Leaf 5)         15                 5         sqrt 0.505294  1.110435 0.996418
        Deep (Depth 20, Leaf 2)         20                 2         sqrt 0.394834  0.954326 0.997354
Half-Features (Depth 15, m=0.5)         15                 5          0.5 0.153795  0.592637 0.998980


Saved figure: /Users/himanshupandey/Desktop/water_quality_prediction/reports/figures/rf_hyperparameter_tuning.png


## 4. Systematic Ablation Matrix (8/8, 7/8, 6/8, 4/8 & Raw vs Engineered)

Parth's experimental protocol specifies evaluating model resilience across measurement scarcity and assessing the value of derived hydrochemical feature engineering.

We compare:
1. **Raw Feature Suite**: The available raw physical/chemical measurements + non-chemical metadata (`Country`, `Waterbody Type`).
2. **Engineered Feature Suite**: Raw measurements + stoichiometric ratios (`N_P_ratio`, `Inorganic_N_ratio`, `Organic_N_estimate`, `Organic_N_fraction`, `BOD_DO_ratio`) + monotonic logs + temporal cyclic features (`sin_day`, `cos_day`, `month`).
3. **Four Measurement Scenarios**:
   - `8/8`: Complete standard Ireland suite.
   - `7/8`: Orthophosphate unmeasured (the dominant CCME excursion contributor).
   - `6/8`: Orthophosphate and BOD unmeasured.
   - `4/8`: Severe missingness: only pH, DO, Temperature, and Nitrate measured.

When a measurement is missing, all dependent derived features are automatically masked via `src.preprocess.mask_features()`.


In [4]:
scenarios = ["8/8", "7/8", "6/8", "4/8"]
ablation_results = []

for sc in scenarios:
    # Raw features
    res_raw = train(
        regime=1,
        feature_set="raw",
        partial_scenario=sc,
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=5,
        subsample=50_000,
        save_reports=False,
        n_jobs=-1,
    )
    ablation_results.append({
        "Scenario": sc,
        "Feature Set": "Raw Features",
        "Features (p)": res_raw["config"]["n_features"],
        "Val MAE": res_raw["val_metrics"]["mae"],
        "Val RMSE": res_raw["val_metrics"]["rmse"],
        "Val R²": res_raw["val_metrics"]["r2"],
    })

    # Engineered features
    res_eng = train(
        regime=1,
        feature_set="engineered",
        partial_scenario=sc,
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=5,
        subsample=50_000,
        save_reports=False,
        n_jobs=-1,
    )
    ablation_results.append({
        "Scenario": sc,
        "Feature Set": "Engineered Features",
        "Features (p)": res_eng["config"]["n_features"],
        "Val MAE": res_eng["val_metrics"]["mae"],
        "Val RMSE": res_eng["val_metrics"]["rmse"],
        "Val R²": res_eng["val_metrics"]["r2"],
    })

df_ablation = pd.DataFrame(ablation_results)
print("Systematic Ablation Matrix (Validation Performance):")
print(df_ablation.to_string(index=False))

# Grouped bar chart comparing RMSE
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=df_ablation,
    x="Scenario",
    y="Val RMSE",
    hue="Feature Set",
    palette=["#4575b4", "#d73027"],
    ax=ax,
)
ax.set_title("Random Forest Validation RMSE: Raw vs. Engineered across Scenarios", fontweight="bold")
ax.set_ylabel("Validation RMSE (lower is better)")
ax.set_xlabel("Measurement Scenario")
ax.legend(title="Feature Configuration")
plt.tight_layout()
plt.savefig(FIGURES / "rf_ablation_comparison.png", dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved figure: {FIGURES / 'rf_ablation_comparison.png'}")


Systematic Ablation Matrix (Validation Performance):
Scenario         Feature Set  Features (p)  Val MAE  Val RMSE   Val R²
     8/8        Raw Features            25 1.124294  1.966505 0.988767
     8/8 Engineered Features            38 0.505294  1.110435 0.996418
     7/8        Raw Features            24 5.875318  8.556519 0.787324
     7/8 Engineered Features            35 5.439711  8.288803 0.800425
     6/8        Raw Features            23 6.181619  8.867306 0.771594
     6/8 Engineered Features            32 5.755284  8.660623 0.782118
     4/8        Raw Features            21 9.256403 12.732880 0.529048
     4/8 Engineered Features            25 9.324799 12.841175 0.521003
Saved figure: /Users/himanshupandey/Desktop/water_quality_prediction/reports/figures/rf_ablation_comparison.png


## 5. Feature Importance: MDI vs. Permutation Importance (Strobl et al., 2007)

In ensemble modeling, interpreting feature contributions requires methodological care.

### 5.1 Mean Decrease in Impurity (MDI) Bias
The default feature importance reported by decision tree ensembles is **Mean Decrease in Impurity (MDI / Gini Importance)**:
$$\text{MDI}(X_j) = \frac{1}{B} \sum_{b=1}^B \sum_{t \in T_b, v(s_t) = X_j} p(t) \Delta i(s_t, t)$$
where $p(t)$ is the proportion of samples reaching node $t$, and $\Delta i(s_t, t)$ is the impurity reduction from splitting on $X_j$.

**Known Flaw (Strobl et al., 2007):**
As proven by Strobl, Boulesteix, Zeileis, and Hothorn (2007, *BMC Bioinformatics*), MDI suffers from severe selection bias:
1. **Variance & Cardinality Bias**: Continuous features with many unique values provide more split candidates, giving them an artificial combinatorial advantage over discrete features.
2. **Collinear Masking**: When two features are correlated (e.g., `Nitrogen` and `Nitrate`), MDI splits credit arbitrarily between them, masking their true independent physical importance.

### 5.2 Permutation Feature Importance
To obtain an unbiased ranking, we compute **Permutation Importance** on validation data:
For feature $j$, we randomly permute its values in $X_{\text{val}}$, breaking its relationship with $y_{\text{val}}$, and compute the resulting increase in prediction error:
$$\text{Importance}(X_j) = \text{RMSE}(X_{\text{val}}^{\pi_j}, y_{\text{val}}) - \text{RMSE}(X_{\text{val}}, y_{\text{val}})$$
Features whose shuffling causes large jumps in RMSE are genuinely critical to model predictions.


In [5]:
# Extract best baseline model and encoder
rf_model = baseline_res["model"]
encoder = baseline_res["encoder"]
feature_names = baseline_res["feature_names"]

# 1. Mean Decrease in Impurity (MDI)
mdi_importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)

# 2. Permutation Importance on a manageable validation sample (10,000 rows)
_, X_val_raw, _, _, y_val, _ = get_regime_splits(regime=1, partial_scenario="8/8")
rng = np.random.default_rng(SEED)
val_sample_idx = rng.choice(len(X_val_raw), size=10_000, replace=False)
X_val_sample = encoder.transform(X_val_raw.iloc[val_sample_idx])
y_val_sample = y_val.iloc[val_sample_idx]

perm_res = permutation_importance(
    rf_model,
    X_val_sample,
    y_val_sample,
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1,
    scoring="neg_root_mean_squared_error",
)
perm_importances = pd.Series(-perm_res.importances_mean, index=feature_names).sort_values(ascending=False)

# Compare top 10 features
top10_mdi = mdi_importances.head(10)
top10_perm = perm_importances.loc[top10_mdi.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

sns.barplot(x=top10_mdi.values, y=top10_mdi.index, palette="Blues_r", ax=axes[0], hue=top10_mdi.index, legend=False)
axes[0].set_title("MDI (Mean Decrease Impurity)\nBiased toward continuous/collinear variables", fontweight="bold")
axes[0].set_xlabel("Relative Importance")

sns.barplot(x=top10_perm.values, y=top10_perm.index, palette="Reds_r", ax=axes[1], hue=top10_perm.index, legend=False)
axes[1].set_title("Permutation Importance (Validation)\nUnbiased increase in RMSE when shuffled", fontweight="bold")
axes[1].set_xlabel("RMSE Increase (CCME scale)")

plt.suptitle("Feature Importance Comparison: Addressing Strobl et al. (2007) Bias", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / "rf_feature_importance_comparison.png", dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved figure: {FIGURES / 'rf_feature_importance_comparison.png'}")


Saved figure: /Users/himanshupandey/Desktop/water_quality_prediction/reports/figures/rf_feature_importance_comparison.png


## 6. Residual Analysis and the 35.92% Target Ceiling

In [`02_eda.ipynb`](02_eda.ipynb), Shaik Tajuddin discovered that **35.92% of the dataset has `CCME_Values` exactly equal to 100.0**.

### 6.1 The Physical and Mathematical Meaning of the Ceiling
In the CCME Water Quality Index formula, when all monitored parameters comply with their respective guideline thresholds, the number of failed parameters $F_1 = 0$, frequency $F_2 = 0$, and amplitude excursion $F_3 = 0$. The resulting index evaluates to:
$$\text{CCME\_Values} = 100 - \frac{\sqrt{0^2 + 0^2 + 0^2}}{1.732} = 100.0$$

Water that is cleaner than the standard does not receive bonus points beyond 100.0. Thus, `CCME_Values` represents a **censored distribution** bounded at $[0, 100]$.

### 6.2 Impact on Tree Regressors
Decision tree leaf predictions compute the local sample mean:
$$\hat{y}(x) = \frac{1}{|R_m|} \sum_{i \in R_m} y_i$$
Because all target values satisfy $y_i \le 100.0$, a leaf containing even a single sub-100 sample will predict $\hat{y} < 100.0$. Consequently, standard regression predictions for pristine waters regress toward the conditional mean ($\approx 98 - 99.5$).

This creates an **asymmetric, one-sided residual cluster** ($y - \hat{y} > 0$) for ceiling observations. This is an unavoidable mathematical consequence of predicting a censored continuous target with regression trees, rather than a failure of model fitting.


In [6]:
# Compute validation predictions on a 50,000-sample validation subset
val_subset_idx = rng.choice(len(X_val_raw), size=50_000, replace=False)
X_val_sub = encoder.transform(X_val_raw.iloc[val_subset_idx])
y_val_sub = y_val.iloc[val_subset_idx].values
y_val_pred = rf_model.predict(X_val_sub)

residuals = y_val_sub - y_val_pred
is_ceiling = np.isclose(y_val_sub, 100.0)

# Metrics conditional on target ceiling
sub_metrics = {
    "Sub-population": ["Full Validation Subset", "Below Ceiling (CCME < 100)", "At Ceiling (CCME == 100)"],
    "Sample Count": [len(y_val_sub), int((~is_ceiling).sum()), int(is_ceiling.sum())],
    "Percentage": ["100.0%", f"{(~is_ceiling).mean():.1%}", f"{is_ceiling.mean():.1%}"],
    "MAE": [
        compute_metrics(y_val_sub, y_val_pred)["mae"],
        compute_metrics(y_val_sub[~is_ceiling], y_val_pred[~is_ceiling])["mae"],
        compute_metrics(y_val_sub[is_ceiling], y_val_pred[is_ceiling])["mae"],
    ],
    "RMSE": [
        compute_metrics(y_val_sub, y_val_pred)["rmse"],
        compute_metrics(y_val_sub[~is_ceiling], y_val_pred[~is_ceiling])["rmse"],
        compute_metrics(y_val_sub[is_ceiling], y_val_pred[is_ceiling])["rmse"],
    ],
}
df_sub_metrics = pd.DataFrame(sub_metrics)
print("Performance Conditioned on Target Ceiling:")
print(df_sub_metrics.to_string(index=False))

# Plot residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Ground truth vs Prediction
axes[0].scatter(y_val_sub[~is_ceiling], y_val_pred[~is_ceiling], alpha=0.3, s=10, color="#1f77b4", label="CCME < 100")
axes[0].scatter(y_val_sub[is_ceiling], y_val_pred[is_ceiling], alpha=0.3, s=10, color="#d62728", label="CCME == 100 (Ceiling)")
axes[0].plot([30, 100], [30, 100], "k--", label="Ideal line")
axes[0].set_title("Ground Truth vs. Random Forest Predictions", fontweight="bold")
axes[0].set_xlabel("Actual CCME_Values")
axes[0].set_ylabel("Predicted CCME_Values")
axes[0].legend()

# 2. Residual distribution split
sns.histplot(residuals[~is_ceiling], bins=50, kde=True, color="#1f77b4", ax=axes[1], stat="density", label="CCME < 100")
sns.histplot(residuals[is_ceiling], bins=50, kde=True, color="#d62728", ax=axes[1], stat="density", label="CCME == 100")
axes[1].axvline(0, color="k", linestyle="--")
axes[1].set_title("Residual Distribution (y - ŷ) Split by Ceiling", fontweight="bold")
axes[1].set_xlabel("Residual Error")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "rf_residuals_analysis.png", dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved figure: {FIGURES / 'rf_residuals_analysis.png'}")


Performance Conditioned on Target Ceiling:
            Sub-population  Sample Count Percentage      MAE     RMSE
    Full Validation Subset         50000     100.0% 0.405712 0.924154
Below Ceiling (CCME < 100)         31178      62.4% 0.545456 1.139741
  At Ceiling (CCME == 100)         18822      37.6% 0.174231 0.342075


Saved figure: /Users/himanshupandey/Desktop/water_quality_prediction/reports/figures/rf_residuals_analysis.png


## 7. Final Holdout Evaluation & Export

Having completed validation-based model selection and scientific diagnosis, we evaluate our final optimal Random Forest model **strictly once on the holdout test partition** (`X_test`, 415,804 rows).

The test results are recorded to `reports/rf_results.csv` for the consolidated results table.


In [7]:
# Single evaluation on holdout test set
test_preds = baseline_res["test_metrics"]

final_benchmark = pd.DataFrame([{
    "Model": "Random Forest Baseline (8/8 Engineered)",
    "Regime": "1 (Formula Emulation)",
    "Train Samples": baseline_res["config"]["n_train_samples"],
    "Features": baseline_res["config"]["n_features"],
    "Val RMSE": baseline_res["val_metrics"]["rmse"],
    "Val R²": baseline_res["val_metrics"]["r2"],
    "Test MAE": test_preds["mae"],
    "Test RMSE": test_preds["rmse"],
    "Test R²": test_preds["r2"],
}])

print("=== Final Selected Model: Holdout Test Generalization ===")
print(final_benchmark.to_string(index=False))
print(f"\nResults appended to: {FIGURES.parent / 'rf_results.csv'}")


=== Final Selected Model: Holdout Test Generalization ===
                                  Model                Regime  Train Samples  Features  Val RMSE   Val R²  Test MAE  Test RMSE  Test R²
Random Forest Baseline (8/8 Engineered) 1 (Formula Emulation)         100000        38  0.907523 0.997608  0.407842   0.919717 0.997568

Results appended to: /Users/himanshupandey/Desktop/water_quality_prediction/reports/rf_results.csv
